In [2]:
### Regenerate 1.5 Msun, Z=0.0142 CO white dwarf

from pathlib import Path
import subprocess
import shutil
import os
import re
import time


# ============================================================
# 1. Paths and desired progenitor
# ============================================================

MESA_DIR = Path("/Users/mchau/mesa")
PROJECT_ROOT = Path("/Users/mchau/mesa_tutorial")

MASS_MSUN = 3
INITIAL_Z = 0.0142

SOURCE_DIR = (
    MESA_DIR
    / "star"
    / "test_suite"
    / "make_co_wd"
)

RUN_DIR = (
    PROJECT_ROOT
    / "WHITE_DWARF"
    / "WD"
    / "runs"
    / "CO_WD_3Msun_Z0p0142"
)

print("MESA_DIR:", MESA_DIR)
print("SOURCE_DIR:", SOURCE_DIR)
print("RUN_DIR:", RUN_DIR)


# ============================================================
# 2. Check MESA installation and source test
# ============================================================

if not MESA_DIR.exists():
    raise FileNotFoundError(
        f"MESA directory not found:\n{MESA_DIR}"
    )

if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"MESA make_co_wd test not found:\n{SOURCE_DIR}"
    )

STAR_SOURCE = MESA_DIR / "star" / "test" / "star"

if not STAR_SOURCE.exists():
    raise FileNotFoundError(
        f"Working MESA star executable not found:\n"
        f"{STAR_SOURCE}"
    )

print("✓ MESA installation found")
print("✓ make_co_wd test found")
print("✓ Working star executable found")


# ============================================================
# 3. Create clean run directory
# ============================================================

if RUN_DIR.exists():

    print("\nRemoving previous run directory...")
    shutil.rmtree(RUN_DIR)

print("\nCopying official make_co_wd test...")

shutil.copytree(
    SOURCE_DIR,
    RUN_DIR
)

print("✓ Test copied to:")
print(RUN_DIR)


# ============================================================
# 4. Copy working MESA executable
# ============================================================

STAR_DEST = RUN_DIR / "star"

shutil.copy2(
    STAR_SOURCE,
    STAR_DEST
)

STAR_DEST.chmod(0o755)

print("\n✓ Copied working star executable:")
print(STAR_DEST)


# ============================================================
# 5. Set progenitor mass and metallicity
# ============================================================

common = RUN_DIR / "inlist_common"

text = common.read_text()


def replace_assignment(text, parameter, value):

    pattern = re.compile(
        rf"(^[ \t]*{re.escape(parameter)}[ \t]*=[ \t]*)"
        rf"([-+0-9.eEdD]+)",
        flags=re.MULTILINE
    )

    new_text, n = pattern.subn(
        rf"\g<1>{value}",
        text
    )

    if n != 1:
        raise RuntimeError(
            f"Expected exactly one occurrence of "
            f"{parameter}, found {n}"
        )

    return new_text


# Initial progenitor mass
text = replace_assignment(
    text,
    "initial_mass",
    f"{MASS_MSUN:.6g}"
)


# Initial progenitor metallicity
text = replace_assignment(
    text,
    "initial_z",
    f"{INITIAL_Z:.6g}"
)


# IMPORTANT:
# Do NOT modify Zbase.
# Preserve the official make_co_wd value.
#
# The official test uses:
#
#     Zbase = 0.00142
#
# while the progenitor has:
#
#     initial_z = 0.0142

common.write_text(text)


print("\nUpdated inlist_common:")

for line in common.read_text().splitlines():

    if any(
        x in line
        for x in [
            "initial_mass",
            "initial_z",
            "Zbase"
        ]
    ):

        print(" ", line)

# ============================================================
# 6. Modify MESA inlists for speed and WD settling
# ============================================================

def set_inlist_parameter(path, parameter, value):
    """
    Replace an existing Fortran namelist assignment.
    Requires exactly one occurrence.
    """
    text = path.read_text()

    pattern = re.compile(
        rf"(^[ \t]*{re.escape(parameter)}[ \t]*=[ \t]*)"
        rf"([-+0-9.eEdD]+)",
        flags=re.MULTILINE
    )

    new_text, n = pattern.subn(
        rf"\g<1>{value}",
        text
    )

    if n != 1:
        raise RuntimeError(
            f"{path.name}: expected exactly one occurrence of "
            f"{parameter}, found {n}"
        )

    path.write_text(new_text)


# ------------------------------------------------------------
# 6A. Speed / timestep settings in inlist_common
# ------------------------------------------------------------

common = RUN_DIR / "inlist_common"

set_inlist_parameter(
    common,
    "mesh_delta_coeff",
    "2.0"
)

set_inlist_parameter(
    common,
    "varcontrol_target",
    "2d-3"
)

set_inlist_parameter(
    common,
    "max_years_for_timestep",
    "1d7"
)


# ------------------------------------------------------------
# 6B. Make sure the progenitor parameters are correct
# ------------------------------------------------------------

set_inlist_parameter(
    common,
    "initial_mass",
    f"{MASS_MSUN:.6g}"
)

set_inlist_parameter(
    common,
    "initial_z",
    f"{INITIAL_Z:.6g}"
)


# IMPORTANT:
# Do NOT change Zbase.
# The official make_co_wd test uses Zbase = 0.00142.


# ------------------------------------------------------------
# 6C. Disable element diffusion during WD settling
# ------------------------------------------------------------

settle = RUN_DIR / "inlist_settle"

settle_text = settle.read_text()


# Change existing do_element_diffusion assignment if present
pattern = re.compile(
    r"(^[ \t]*do_element_diffusion[ \t]*=[ \t]*)(\.true\.|\.false\.)",
    flags=re.MULTILINE | re.IGNORECASE
)

settle_text, n = pattern.subn(
    r"\g<1>.false.",
    settle_text
)

if n == 0:

    # If it isn't already present, add it to &controls
    settle_text = settle_text.replace(
        "&controls",
        "&controls\n"
        "      do_element_diffusion = .false.",
        1
    )

elif n > 1:

    raise RuntimeError(
        f"inlist_settle: found {n} do_element_diffusion "
        f"assignments; expected at most one."
    )


settle.write_text(settle_text)


# ------------------------------------------------------------
# 6D. Print what we changed
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("UPDATED inlist_common")
print("=" * 70)

for line in common.read_text().splitlines():

    if any(
        parameter in line
        for parameter in [
            "initial_mass",
            "initial_z",
            "Zbase",
            "mesh_delta_coeff",
            "varcontrol_target",
            "max_years_for_timestep",
        ]
    ):

        print(line)


print("\n" + "=" * 70)
print("UPDATED inlist_settle")
print("=" * 70)

for line in settle.read_text().splitlines():

    if "do_element_diffusion" in line:

        print(line)

print("\n✓ Inlist modifications complete.")

# ============================================================
# Speed up core-He-burning evolution
# ============================================================

he_burn = RUN_DIR / "inlist_to_end_he_core_burn"

he_text = he_burn.read_text()

settings = {
    "mesh_delta_coeff": "2.5",
    "varcontrol_target": "5d-3",
    "max_years_for_timestep": "5d7",
}

for parameter, value in settings.items():

    pattern = re.compile(
        rf"(^[ \t]*{re.escape(parameter)}[ \t]*=[ \t]*)[^\n]*",
        flags=re.MULTILINE
    )

    if pattern.search(he_text):
        he_text = pattern.sub(
            rf"\g<1>{value}",
            he_text,
            count=1
        )
    else:
        he_text = he_text.replace(
            "&controls",
            f"&controls\n      {parameter} = {value}",
            1
        )

he_burn.write_text(he_text)

print("\nUpdated helium-burning settings:")

for line in he_burn.read_text().splitlines():
    if any(
        p in line
        for p in [
            "mesh_delta_coeff",
            "varcontrol_target",
            "max_years_for_timestep",
        ]
    ):
        print(line)

# ============================================================
# 6. Environment
# ============================================================

env = os.environ.copy()

env["MESA_DIR"] = str(MESA_DIR)

print("\nEnvironment:")
print("MESA_DIR =", env["MESA_DIR"])


# ============================================================
# 7. Function to run one MESA stage
# ============================================================

def run_stage(stage_number, stage_name, header_name, output_model):

    print("\n")
    print("=" * 75)
    print(f"STAGE {stage_number}: {stage_name}")
    print("=" * 75)

    header_path = RUN_DIR / header_name
    output_path = RUN_DIR / output_model

    if not header_path.exists():

        raise FileNotFoundError(
            f"Missing inlist header:\n{header_path}"
        )


    # MESA expects the active inlist to be called "inlist"

    shutil.copy2(
        header_path,
        RUN_DIR / "inlist"
    )


    # Log filename

    safe_name = (
        stage_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    log_path = RUN_DIR / f"{safe_name}.log"


    print(f"Header: {header_name}")
    print(f"Expected output: {output_model}")
    print(f"Log file: {log_path}")
    print()


    start = time.time()


    # --------------------------------------------------------
    # Run MESA quietly
    # --------------------------------------------------------

    with open(log_path, "w") as log:

        log.write(
            f"MESA STAGE {stage_number}: {stage_name}\n"
        )

        log.write("=" * 75 + "\n")

        log.write(
            f"Mass = {MASS_MSUN} Msun\n"
        )

        log.write(
            f"Initial Z = {INITIAL_Z}\n"
        )

        log.write(
            f"Header = {header_name}\n"
        )

        log.write(
            f"Expected output = {output_model}\n"
        )

        log.write("=" * 75 + "\n\n")

        log.flush()

        # Send ALL MESA output to the log file.
        # Nothing will be dumped into the Jupyter output.
        result = subprocess.run(
            [str(STAR_DEST)],
            cwd=RUN_DIR,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            text=True,
        )

        return_code = result.returncode


    # --------------------------------------------------------
    # Check whether MESA succeeded
    # --------------------------------------------------------

    if return_code != 0:

        raise RuntimeError(
            f"\nMESA failed during stage:\n"
            f"{stage_name}\n\n"
            f"Return code: {return_code}\n"
            f"Full log:\n{log_path}"
        )


    # --------------------------------------------------------
    # Check output model
    # --------------------------------------------------------

    if not output_path.exists():

        raise FileNotFoundError(
            f"\nMESA returned successfully, "
            f"but expected model was not created:\n"
            f"{output_path}\n\n"
            f"Check log:\n{log_path}"
        )


    size_mb = (
        output_path.stat().st_size
        / (1024 ** 2)
    )


    print(
        f"✓ Created {output_model} "
        f"({size_mb:.2f} MB)"
    )


    return log_path


# ============================================================
# 8. Run the five official stages
# ============================================================

overall_start = time.time()


stages = [

    (
        1,
        "ZAMS",
        "inlist_zams_header",
        "zams.mod",
    ),

    (
        2,
        "End He core burn",
        "inlist_to_end_he_core_burn_header",
        "end_he_core_burn.mod",
    ),

    (
        3,
        "CO core",
        "inlist_co_core_header",
        "co_core.mod",
    ),

    (
        4,
        "Remove envelope",
        "inlist_remove_env_header",
        "co_wd.mod",
    ),

    (
        5,
        "Settle",
        "inlist_settle_header",
        "final.mod",
    ),

]


for stage_number, stage_name, header_name, output_model in stages:

    run_stage(
        stage_number,
        stage_name,
        header_name,
        output_model
    )


overall_elapsed = (
    time.time() - overall_start
) / 60


# ============================================================
# 9. Verify all models
# ============================================================

print("\n")
print("=" * 75)
print("CHECKING GENERATED MODELS")
print("=" * 75)


expected_models = [

    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",

]


for name in expected_models:

    path = RUN_DIR / name

    if path.exists() and path.stat().st_size > 0:

        size_mb = (
            path.stat().st_size
            / (1024 ** 2)
        )

        print(
            f"✓ {name:<25} "
            f"{size_mb:>8.2f} MB"
        )

    else:

        print(
            f"✗ {name:<25} MISSING"
        )


# ============================================================
# 10. Verify CO-core checkpoint
# ============================================================

co_core = RUN_DIR / "co_core.mod"

if not co_core.exists():

    raise FileNotFoundError(
        "co_core.mod does not exist."
    )


header = "\n".join(
    co_core
    .read_text(errors="replace")
    .splitlines()[:30]
)


print("\n")
print("=" * 75)
print("CO-CORE MODEL HEADER")
print("=" * 75)

print(header)


# Extract mass

mass_match = re.search(
    r"M/Msun\s+([0-9.EeDd+-]+)",
    header
)


# Extract initial metallicity

z_match = re.search(
    r"initial_z\s+([0-9.EeDd+-]+)",
    header
)


if mass_match:

    generated_mass = float(
        mass_match
        .group(1)
        .replace("D", "E")
    )

    print(
        "\nGenerated mass:",
        generated_mass,
        "Msun"
    )

    print(
        "Requested mass:",
        MASS_MSUN,
        "Msun"
    )


if z_match:

    generated_z = float(
        z_match
        .group(1)
        .replace("D", "E")
    )

    print(
        "Generated initial Z:",
        generated_z
    )

    print(
        "Requested initial Z:",
        INITIAL_Z
    )


    if abs(generated_z - INITIAL_Z) > 1e-10:

        raise RuntimeError(
            f"Metallicity mismatch!\n"
            f"Expected: {INITIAL_Z}\n"
            f"Got: {generated_z}"
        )


print("\n✓ CO-core metallicity verified")


# ============================================================
# 11. List log files
# ============================================================

print("\n")
print("=" * 75)
print("LOG FILES")
print("=" * 75)


for path in sorted(RUN_DIR.glob("*.log")):

    size_kb = (
        path.stat().st_size
        / 1024
    )

    print(
        f"{path.name:<35} "
        f"{size_kb:>10.1f} KB"
    )


# ============================================================
# 12. Final summary
# ============================================================

print("\n")
print("=" * 75)
print("MESA CO-WD MODEL CREATION COMPLETE")
print("=" * 75)

print(
    f"Progenitor mass: {MASS_MSUN} Msun"
)

print(
    f"Initial metallicity: Z = {INITIAL_Z}"
)

print(
    f"Total runtime: {overall_elapsed:.2f} minutes"
)

print(
    f"\nRun directory:\n{RUN_DIR}"
)

print(
    "\nImportant files:"
)

for name in [
    "zams.mod",
    "end_he_core_burn.mod",
    "co_core.mod",
    "co_wd.mod",
    "final.mod",
]:

    print(
        f"  {RUN_DIR / name}"
    )

print(
    "\n✓ Finished."
)

MESA_DIR: /Users/mchau/mesa
SOURCE_DIR: /Users/mchau/mesa/star/test_suite/make_co_wd
RUN_DIR: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142
✓ MESA installation found
✓ make_co_wd test found
✓ Working star executable found

Copying official make_co_wd test...
✓ Test copied to:
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142

✓ Copied working star executable:
/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/star

Updated inlist_common:
        Zbase = 0.02d0
        initial_mass = 3 ! this default produces a ~0.6 Msun WD.
        ! initial_mass in the range 1.0-6.5 Msun will produce CO WDs
        initial_z = 0.0142

UPDATED inlist_common
      Zbase = 0.02d0
      initial_mass = 3 ! this default produces a ~0.6 Msun WD.
      ! initial_mass in the range 1.0-6.5 Msun will produce CO WDs
      initial_z = 0.0142
      mesh_delta_coeff = 2.0
      max_years_for_timestep = 1d7
      varcontrol_target = 2d-3

UPDATED inlist_settle
   

KeyboardInterrupt: 

In [4]:
from pathlib import Path
import subprocess
import os

RUN_DIR = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/"
    "CO_WD_3Msun_Z0p0142"
)

TEST_DIR = Path("/Users/mchau/mesa/star/test_suite/make_co_wd")
STAR = TEST_DIR / "star"

# Initialize MESA environment
env = os.environ.copy()
env["MESA_DIR"] = "/Users/mchau/mesa"
env["MESASDK_ROOT"] = "/Applications/mesasdk"
env["PATH"] = f"/Applications/mesasdk/bin:{env['PATH']}"

# Copy the official remove_env header
(RUN_DIR / "inlist").write_text(
    (TEST_DIR / "inlist_remove_env_header").read_text()
)

# Log all MESA output without displaying it in Jupyter
log_path = RUN_DIR / "remove_env.log"

print("Running remove_env...")
print(f"Log: {log_path}")

with log_path.open("w") as log:

    result = subprocess.run(
        [str(STAR)],
        cwd=RUN_DIR,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

return_code = result.returncode

if return_code != 0:
    print("✗ remove_env FAILED")
    print(f"Return code: {return_code}")
    print(f"See full output in: {log_path}")
    raise RuntimeError(
        f"remove_env failed with return code {return_code}"
    )

print("✓ remove_env completed successfully.")
print(f"Full MESA output saved to: {log_path}")

Running remove_env...
Log: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/remove_env.log
✓ remove_env completed successfully.
Full MESA output saved to: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/remove_env.log


In [5]:
from pathlib import Path
import numpy as np

history = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/"
    "CO_WD_3Msun_Z0p0142/LOGS/history.data"
)

# Find the header and data
lines = history.read_text().splitlines()

header = None
data = []

for line in lines:
    if line.strip().startswith("model_number"):
        header = line.split()
    elif header is not None and line.strip():
        vals = line.split()
        if len(vals) == len(header):
            data.append(vals)

# Final model
row = dict(zip(header, data[-1]))

for key in [
    "model_number",
    "star_mass",
    "he_core_mass",
    "co_core_mass",
    "total_mass_h1",
    "total_mass_he4",
    "log_Teff",
    "log_R",
]:
    print(f"{key:20s} {row[key]}")

model_number         946
star_mass            6.0252634723676435E-001
he_core_mass         6.0151858007203141E-001
co_core_mass         5.7251917361044102E-001
total_mass_h1        6.5787782998806121E-004
total_mass_he4       1.9956353045104916E-002
log_Teff             4.8026674679257670E+000
log_R                -3.5057521558889349E-001


In [6]:
from pathlib import Path
import subprocess
import os

RUN_DIR = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/"
    "CO_WD_3Msun_Z0p0142"
)

TEST_DIR = Path("/Users/mchau/mesa/star/test_suite/make_co_wd")
STAR = TEST_DIR / "star"

# MESA environment
env = os.environ.copy()
env["MESA_DIR"] = "/Users/mchau/mesa"
env["MESASDK_ROOT"] = "/Applications/mesasdk"
env["PATH"] = f"/Applications/mesasdk/bin:{env['PATH']}"

# Use the official settle header as the main inlist
(RUN_DIR / "inlist").write_text(
    (TEST_DIR / "inlist_settle_header").read_text()
)

# Run settle
process = subprocess.Popen(
    [str(STAR)],
    cwd=RUN_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

with (RUN_DIR / "settle.log").open("w") as log:
    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"settle failed with return code {return_code}"
    )

print("\nsettle completed successfully.")

 reading user weak rate file /Users/mchau/mesa/data/rates_data/rate_tables/S13_r_be7_wk_li7.h5
 version_number r24.08.1
 read inlist_common
 read inlist_settle
load saved model co_wd.mod

 set special rate factor for r_c12_ag_o16           1   1.0000000000000000     
 set special rate factor for r_he4_he4_he4_to_c12          10   1.0000000000000000     
create rate data for r_c12_ag_o16
create rate data for r_o16_ga_c12
create rate data for r_c12_to_he4_he4_he4
create rate data for r_he4_he4_he4_to_c12
write /Users/mchau/mesa/data/rates_data/cache/r_c12_ag_o16_0.bin
write /Users/mchau/mesa/data/rates_data/cache/r_o16_ga_c12_0.bin
write /Users/mchau/mesa/data/rates_data/cache/r_c12_to_he4_he4_he4_0.bin
write /Users/mchau/mesa/data/rates_data/cache/r_he4_he4_he4_to_c12_0.bin
 set special rate factor for r_c12_ag_o16           1   1.0000000000000000     
 set special rate factor for r_he4_he4_he4_to_c12          10   1.0000000000000000     
                             set_initial_number_

In [8]:
from pathlib import Path

history = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/"
    "CO_WD_3Msun_Z0p0142/LOGS/history.data"
)

lines = history.read_text().splitlines()

header = None
data = []

for line in lines:
    if line.strip().startswith("model_number"):
        header = line.split()
    elif header is not None and line.strip():
        vals = line.split()
        if len(vals) == len(header):
            data.append(vals)

row = dict(zip(header, data[-1]))

for key in [
    "model_number",
    "star_age",
    "star_mass",
    "he_core_mass",
    "co_core_mass",
    "total_mass_h1",
    "total_mass_he4",
    "log_Teff",
    "log_R",
    "log_L",
]:
    print(f"{key:20s} {row.get(key)}")

model_number         1345
star_age             4.4649924180806661E+008
star_mass            6.0252634723676435E-001
he_core_mass         6.0240468842889072E-001
co_core_mass         5.8173710993061400E-001
total_mass_h1        6.5403514054797966E-005
total_mass_he4       1.4587778518300480E-002
log_Teff             4.6667750705486437E+000
log_R                -1.8152431815989734E+000
log_L                -8.6924027323635137E-003


In [13]:
from pathlib import Path
import subprocess
import os

run_dir = Path(
    "/Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/"
    "CO_WD_3Msun_Z0p0142"
)

export_dir = run_dir / "gyre_export"
export_dir.mkdir(exist_ok=True)

# ------------------------------------------------------------
# MESA environment
# ------------------------------------------------------------

env = os.environ.copy()
env["MESA_DIR"] = "/Users/mchau/mesa"
env["MESASDK_ROOT"] = "/Applications/mesasdk"
env["PATH"] = f"/Applications/mesasdk/bin:{env['PATH']}"

# ------------------------------------------------------------
# Export inlist
# ------------------------------------------------------------

inlist = """\
&star_job
      load_saved_model = .true.
      load_model_filename = '../final.mod'

      write_profile_when_terminate = .true.
      filename_for_profile_when_terminate = 'profile.data'
/

&eos
/

&kap
      Zbase = 0.02d0
/

&controls
      max_model_number = 1

      write_pulse_data_with_profile = .true.
      pulse_data_format = 'GYRE'
/
"""

(export_dir / "inlist").write_text(inlist)

# ------------------------------------------------------------
# Run MESA quietly
# ------------------------------------------------------------

star = Path("/Users/mchau/mesa/star/test_suite/make_co_wd/star")
log_path = export_dir / "export.log"

print("Converting final.mod to GYRE format...")
print(f"Input:  {run_dir / 'final.mod'}")
print(f"Output: {export_dir / 'profile.data.GYRE'}")

with log_path.open("w") as log:
    result = subprocess.run(
        [str(star), "inlist"],
        cwd=export_dir,
        env=env,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

# ------------------------------------------------------------
# Check result
# ------------------------------------------------------------

print(f"\nReturn code: {result.returncode}")

gyre_file = export_dir / "profile.data.GYRE"
mesa_profile = export_dir / "profile.data"

if gyre_file.exists():
    print("✓ GYRE file created:")
    print(f"  {gyre_file}")
else:
    print("✗ GYRE file was NOT created.")
    print(f"Check MESA output: {log_path}")

if mesa_profile.exists():
    print("✓ MESA profile created:")
    print(f"  {mesa_profile}")

print(f"\nFull MESA output saved to:")
print(f"  {log_path}")

Converting final.mod to GYRE format...
Input:  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/final.mod
Output: /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/gyre_export/profile.data.GYRE

Return code: 0
✓ GYRE file created:
  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/gyre_export/profile.data.GYRE
✓ MESA profile created:
  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/gyre_export/profile.data

Full MESA output saved to:
  /Users/mchau/mesa_tutorial/WHITE_DWARF/WD/runs/CO_WD_3Msun_Z0p0142/gyre_export/export.log
